# Morning Drill — three agents, one student

    prepare -> plan(progress agent) -> author(trainer agent) -> present
                                                                   |
                                     +-----------------------------+
                                     v
                              await_submission  <--- you, via interrupt()
                                     |
                                  verify   (arithmetic, not opinion)
                                     |
                                  grade    (grader agent reviews the CODE)
                                     |
                                  record
                                     |
                          passed? --+-- no --> back to await_submission
                                     |
                                    yes --> END, and the next run starts over

**Progress agent** reads `progress.json` and the session so far, judges the
challenge you just finished, and briefs the trainer agent on what to set next
— including when to quietly fold an older skill back in.

**Trainer agent** writes the challenge: a prose mission for you, and a
declarative spec for the machine. It never writes grading code.

**Grader agent** reviews your code — constraints, reasoning, idiom, what you
did well. Whether the *answer* is right is computed from the spec before the
grader agent is asked anything, and it cannot overturn that.

Without a key in `.env` all three fall back to rules and a hand-written
challenge library, and everything below still runs.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PROJECT.md").exists())
sys.path.insert(0, str(ROOT))

from morning_drill import Session
from morning_drill.llm import coach_status

print(coach_status())

## 1. Start

The progress agent decides, the trainer agent writes, the challenge is
validated, and only then does anything get printed.

In [ ]:
session = Session()      # agents=False for the rules-only version
df = session.start()

## 2. Explore

In [ ]:
df.head()

In [ ]:
df.groupby("disease")["intensity"].describe()

## 3. Submit

The cell that calls `submit()` is captured automatically, so the grader agent
reviews the code you actually wrote, not just the value it produced.

In [ ]:
answer = (
    df[df["qc_score"] >= 0.85]
    .groupby("disease")["intensity"]
    .median()
)

session.submit(answer)

Keep going in a new cell. The challenge is the same one — the retry loop
lives inside the graph, and the hint gets stronger each attempt rather than
restating itself.

In [ ]:
answer = ...

session.submit(answer)

## 4. Things that do not count as a pass

A right answer is not enough. These are all rejected, and none of them needs
a model to catch:

- typing the answer in as a literal instead of deriving it
- a `for` loop or comprehension where the mission said to vectorize
- `.apply()` where a vectorized operation exists, when the mission forbade it

Try one:

In [ ]:
session.submit("AD", code='answer = "AD"')

## 5. Progress and the reference

`reference()` refuses until you have solved it.

In [ ]:
session.reference()

In [ ]:
session.status()

## 6. Ask without submitting

Goes to the grader agent with the mission as context. It will not hand over
the solution, however the question is phrased.

In [ ]:
session.ask("when is idxmax the wrong tool for finding a group?")

## 7. Under the hood

What the trainer agent actually produced, and how good the challenge is.

`discrimination` is the check that stopped the original drill's weakness from
recurring: perturb the spec the way a confused student would — wrong
statistic, dropped filter — and require that at least one perturbation
changes the answer. A challenge nobody can fail for the wrong reason is not
teaching anything, and the trainer agent is sent back to revise it.

In [ ]:
import json

print(json.dumps(session.challenge["spec"], indent=1))
print()
print(session._last_state["discrimination"])
print()
print("trainer revisions:", session._last_state["trainer_notes"] or "none needed")

## 8. The graph

In [ ]:
from morning_drill.session import build_session_graph

print(build_session_graph().get_graph().draw_mermaid())

## 9. The log

One line per attempt, in `logs/session.jsonl`. This is the raw material for
noticing which concepts keep needing hints.

In [ ]:
log = ROOT / "logs" / "session.jsonl"

if log.exists():
    for line in log.read_text().strip().splitlines()[-8:]:
        r = json.loads(line)
        print(f"{r['at']}  {r['topic']:<12} L{r['level']}  "
              f"{'pass' if r['passed'] else 'fail'}  {r.get('error_class') or ''}")